# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json), which provides ordered logistic regression results pertaining to the adoption of indigenous and modern knowledge in rangeland management practices in Northern Kenya (Samburu, Isiolo, Marsabit counties).

We use the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library for easy loading and programmatic exploration, referencing all entities (record sets, fields/columns, etc.) *by their `@id`*.

### Dataset Source
The Croissant schema describing this dataset is available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install mlcroissant library if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load the metadata and all record sets from the dataset using `mlcroissant`. We'll use the Croissant schema URL and inspect the top-level dataset metadata. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display summary of the dataset's metadata
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview

Explore the available record sets and fields. All entities are referenced via their `@id`. Let's list the available record set `@id`s and, for each, list a summary of available field `@id`s and column names. 

In [ ]:
# List all available record sets by their @id
record_set_ids = [r['@id'] for r in dataset.as_json_dict().get('recordSet', [])]
print(f"Record Set @ids found: {record_set_ids if record_set_ids else 'No record sets found!'}")

# For demonstration, let's explore the fields for each record set (if any exist)
if record_set_ids:
    for rec_id in record_set_ids:
        rs = dataset.record_set(rec_id)
        print(f"\nRecord Set: {rec_id}")
        # List fields and their @id for this recordset, if available
        if hasattr(rs, 'field') and rs.field:
            field_ids = [f['@id'] for f in rs.field]
            print(f"Fields: {field_ids}")
            col_names = [f.get('column', {}).get('@id', 'N/A') for f in rs.field]
            print(f"Columns: {col_names}")
        else:
            print("(No field definitions found)")
else:
    print("No structured record sets present. Try dataset.records() or see dataset.metadata for available files.")

## 3. Data Extraction

Extract actual tabular records from the available record sets. All record set and field `@id`s are used for referencing. If present, load each record set into a Pandas DataFrame for analysis.

In [ ]:
# Extract data from each record set (using their @id)

dataframes = dict()
if not record_set_ids:
    print("No record sets to extract. Please check the dataset structure.")
else:
    for rec_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rec_id))
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            print(f"Loaded {len(df)} records from record set {rec_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"Could not load data for {rec_id}: {e}")

# If specific record sets and fields to focus on, you can set them here explicitly
# Example: record_set_to_explore = '<@id_of_key_record_set>'
if dataframes:
    example_record_set_id = next(iter(dataframes.keys()))  # Take first as demo
    print(f"Example record set for further exploration: {example_record_set_id}")
    print("First rows:")
    display(dataframes[example_record_set_id].head())
else:
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Apply some data exploration steps on a numeric field using the selected record set. This example shows how to filter, normalize, and group, referencing fields by their `@id`. Please modify `<numeric_field_id>` and `<group_field_id>` as appropriate for your valid data fields. 

In [ ]:
# Demonstrate filtering and normalization using one record set (update @id as needed)

if example_record_set_id and not dataframes[example_record_set_id].empty:
    df = dataframes[example_record_set_id]
    print(f"Columns in selected record set {example_record_set_id}: {df.columns.tolist()}")
    # Try to auto-select a numeric column (or set one explicitly)
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_field_candidates:
        print('No numeric fields found for EDA.')
    else:
        numeric_field_id = numeric_field_candidates[0]  # For demo
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.8)  # Take 80th percentile as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a categorical/grouping field
        group_field_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field_id = group_field_candidates[0] if group_field_candidates else None
        if group_field_id:
            print(f"Grouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (showing means):")
            display(grouped_df.head())
        else:
            print('No grouping/categorical field found.')
else:
    print('No data available for EDA. Please check the extracted data.')

## 5. Visualization

Visualize numeric data distributions and relationships in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize if data is available
if example_record_set_id and not dataframes[example_record_set_id].empty:
    df = dataframes[example_record_set_id]
    # Try to find a numeric field to plot
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='skyblue')
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
    else:
        print("No numeric fields found to plot.")
else:
    print('No data available for visualization.')

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset using the `mlcroissant` library, explored available record sets and fields by their `@id`, extracted records into data frames, conducted basic filtering and normalization for a numeric field, and visualized data distributions. This approach makes it easy to programmatically access and analyze FAIR-compliant datasets and can be adapted to a wide variety of Croissant-structured resources.